In [62]:
fun <T> (Iterable<T>).toPairs(): Sequence<Pair<T, T>> = sequence {
    val list = this@toPairs.toList()
    for (i in 0 until list.size) {
        for (j in 0 until list.size) {
            yield(list[i] to list[j])
        }
    }
}

In [63]:
class Species(
    val name: String,
    val baseGrowthFn: (Int, Double) -> Double,
    val baseDeathFn: (Int) -> Double,
    val biomass: Double,
    val carryingCapacityClass: String? = null,
    val canConsume: ((Species) -> Double?)? = null,
) {
    companion object {
        fun producer(
            name: String,
            carryingCapacityClass: String,
            growthRate: Double,
            deathRate: Double,
            biomass: Double
        ) = Species(
            name,
            { population, percentCapacity -> population * growthRate * (1 - percentCapacity) },
            { population -> population * deathRate },
            biomass,
            carryingCapacityClass
        )

        fun consumer(
            name: String,
            growthRate: Double,
            deathRate: Double,
            biomass: Double,
            canConsume: (Species) -> Double?
        ) = Species(
            name,
            { population, percentFood -> population * growthRate * percentFood },
            { population -> population * deathRate },
            biomass,
            null,
            canConsume
        )
    }
}
// I probably need a trophic energy loss function

fun consumeFromMap(map: Map<String, Double>) = { species: Species -> map[species.name] }

val grass = Species.producer("grass", "carpet", 0.01, 0.03, 2.72)
val hare = Species.consumer("hare", 5.0, 0.017, 4.0, consumeFromMap(mapOf("grass" to 7.25)))

In [78]:
class Ecosystem(var populations: Map<Species, Int>, var carryingCapacities: Map<String, Double>)

fun tickEcosystem(ecosystem: Ecosystem): Ecosystem {
    val desiredConsumptionIndividuals =
        ecosystem.populations.keys.toPairs()
            .filter { (speciesA, speciesB) -> speciesA.canConsume != null && speciesA.canConsume!!(speciesB) != null }
            .associateWith { (speciesA, speciesB) ->
                ecosystem.populations[speciesA]!! * (speciesA.canConsume!!(speciesB) ?: 0.0) / speciesB.biomass
            }

    println("pairs: ${ecosystem.populations.keys.toPairs().toList().map { (a, b) -> "${a.name} -> ${b.name}" }}")
    println("desiredConsumption: ${desiredConsumptionIndividuals.mapKeys { it.key.first.name to it.key.second.name }}")

    val predationImpact = desiredConsumptionIndividuals.keys
        .groupBy { it.second }
        .mapValues { (_, species) -> species.sumOf { desiredConsumptionIndividuals[it]!! } }
    val overPredationAdjustment = predationImpact.mapValues { (species, desiredConsumption) ->
        min(1.0, ecosystem.populations[species]!! / desiredConsumption)
    }

    val possibleConsumptionIndividuals = desiredConsumptionIndividuals
        .mapValues { (pair, desiredConsumption) -> desiredConsumption * overPredationAdjustment[pair.second]!! }

    println("possibleConsumption: ${possibleConsumptionIndividuals.mapKeys { it.key.first.name to it.key.second.name }}")

    val consumptionGrowthFactors = possibleConsumptionIndividuals.keys
        .groupBy { it.first }
        .mapValues { (_, species) ->
            species.sumOf {
                possibleConsumptionIndividuals[it]!! * it.second.biomass / it.first.canConsume!!(
                    it.second
                )!! / ecosystem.populations[it.first]!!
            }
        }


    println("consumptionGrowthFactors: ${possibleConsumptionIndividuals.mapKeys { it.key.first.name to it.key.second.name }}")

    val actualConsumptionIndividuals =
        possibleConsumptionIndividuals.mapValues { (pair, possibleConsumption) ->
            possibleConsumption / maxOf(
                1.0,
                consumptionGrowthFactors[pair.first]!!
            )
        }

    val growthAmounts = ecosystem.populations.mapValues { (species, population) ->
        species.baseGrowthFn(
            population,
            if (species.carryingCapacityClass != null) population / ecosystem.carryingCapacities[species.carryingCapacityClass]!!
            else (consumptionGrowthFactors[species] ?: 0.0).coerceIn(0.0, 1.0)
        ).roundToInt()
    }
    val naturalDeathAmounts =
        ecosystem.populations.mapValues { (species, population) -> species.baseDeathFn(population).roundToInt() }
    val predationDeathAmounts = actualConsumptionIndividuals.keys
        .groupBy { it.second }
        .mapValues { (_, species) -> species.sumOf { actualConsumptionIndividuals[it]!! }.roundToInt() }

    println("growth: ${growthAmounts.mapKeys { it.key.name }}")
    println("death: ${naturalDeathAmounts.mapKeys { it.key.name }}")
    println("predation: ${predationDeathAmounts.mapKeys { it.key.name }}")

    return Ecosystem(
        ecosystem.populations.mapValues { (species, population) ->
            maxOf(
                0,
                population + growthAmounts[species]!! - naturalDeathAmounts[species]!! - (predationDeathAmounts[species] ?: 0)
            )
        },
        ecosystem.carryingCapacities
    )
}

In [79]:
val testEcosystem = Ecosystem(
    mapOf(
        grass to 1000,
        hare to 10
    ),
    mapOf("carpet" to 2000.0)
)
(1..10).fold(testEcosystem) { ecosystem, _ ->
    val nextEcosystem = tickEcosystem(ecosystem)
    nextEcosystem.populations.forEach {
        println("${it.key.name}: ${it.value}")
    }
    nextEcosystem
}

pairs: [grass -> grass, grass -> hare, hare -> grass, hare -> hare]
desiredConsumption: {(hare, grass)=26.65441176470588}
possibleConsumption: {(hare, grass)=26.65441176470588}
consumptionGrowthFactors: {(hare, grass)=26.65441176470588}
growth: {grass=5, hare=50}
death: {grass=30, hare=0}
predation: {grass=27}
grass: 948
hare: 60
pairs: [grass -> grass, grass -> hare, hare -> grass, hare -> hare]
desiredConsumption: {(hare, grass)=159.92647058823528}
possibleConsumption: {(hare, grass)=159.92647058823528}
consumptionGrowthFactors: {(hare, grass)=159.92647058823528}
growth: {grass=5, hare=300}
death: {grass=28, hare=1}
predation: {grass=160}
grass: 765
hare: 359
pairs: [grass -> grass, grass -> hare, hare -> grass, hare -> hare]
desiredConsumption: {(hare, grass)=956.8933823529411}
possibleConsumption: {(hare, grass)=765.0}
consumptionGrowthFactors: {(hare, grass)=765.0}
growth: {grass=5, hare=1435}
death: {grass=23, hare=6}
predation: {grass=765}
grass: 0
hare: 1788
pairs: [grass -> gr

Line_80_jupyter$Ecosystem@11434c07